In [1]:
import torch
import torch.nn as nn
from model import Transformer
from config import get_config, get_weight_file_path
from train import get_dataset, get_model, greedy_decode
import altair as alt
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

/Users/ngkuissi/miniforge3/envs/transformer/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device {device}")

Using device mps


In [3]:
config = get_config()
train_dataloader, val_dataloader, src_tokenizer, tgt_tokenizer = get_dataset(config)
model = get_model(config, vocab_src_length=src_tokenizer.get_vocab_size(),
                  target_src_length=tgt_tokenizer.get_vocab_size())
model_path = "../opus_books_weights/torch_model.pt"
state = torch.load(model_path, map_location="cpu")
model.load_state_dict(state['model_state_dict'])


Max length of source sentence: 471
Max length of target sentence: 482


<All keys matched successfully>

In [4]:
def load_next_batch(val_dataloader=val_dataloader, device=device, src_tokenizer=src_tokenizer, tgt_tokenizer=tgt_tokenizer):
    batch = next(iter(val_dataloader))
    encoder_input = batch["encoder_input"].to(device) #(batch_size, seq_len)
    decoder_input = batch["decoder_input"].to(device) #(batch_size, seq_len)
    
    encoder_input_tokens = [src_tokenizer.id_to_token(idx) for idx in encoder_input[0].cpu().numpy()]
    decoder_input_tokens = [tgt_tokenizer.id_to_token(idx) for idx in decoder_input[0].cpu().numpy()]
    
    return batch, encoder_input_tokens, decoder_input_tokens

In [32]:
def mtx2df(m, max_row, max_col, row_tokens, col_tokens):
    return pd.DataFrame(
        [
            (
                r,
                c,
                float(m[r, c]),
                "%.3d %s" % (r, row_tokens[r] if len(row_tokens) > r else "<blank>"),
                "%.3d %s" % (c, col_tokens[c] if len(col_tokens) > c else "<blank>"),
            )
            for r in range(m.shape[0])
            for c in range(m.shape[1])
            if r < max_row and c < max_col
        ],
        columns=["row", "column", "value", "row_token", "col_token"],
    )

def get_attn_map(attn_type: str, layer: int, head: int, model=model):
    if attn_type == "encoder":
        attn = model.encoder.layers[layer].attention_block.attention_scores
    elif attn_type == "decoder":
        attn = model.decoder.layers[layer].self_attention_block.attention_scores
    elif attn_type == "encoder-decoder":
        attn = model.decoder.layers[layer].cross_attention_block.attention_scores
    return attn[0, head].data

def attn_map(attn_type, layer, head, row_tokens, col_tokens, max_sentence_len):
    df = mtx2df(
        get_attn_map(attn_type, layer, head),
        max_sentence_len,
        max_sentence_len,
        row_tokens,
        col_tokens,
    )
    return (
        alt.Chart(data=df)
        .mark_rect()
        .encode(
            x=alt.X("col_token", axis=alt.Axis(title="")),
            y=alt.Y("row_token", axis=alt.Axis(title="")),
            color="value",
            tooltip=["row", "column", "value", "row_token", "col_token"],
        )
        #.title(f"Layer {layer} Head {head}")
        .properties(height=400, width=400, title=f"Layer {layer} Head {head}")
        .interactive()
    )

def get_all_attention_maps(attn_type: str, layers: list[int], heads: list[int], row_tokens: list, col_tokens, max_sentence_len: int):
    charts = []
    for layer in layers:
        rowCharts = []
        for head in heads:
            rowCharts.append(attn_map(attn_type, layer, head, row_tokens, col_tokens, max_sentence_len))
        charts.append(alt.hconcat(*rowCharts))
    return alt.vconcat(*charts)

In [33]:
batch, encoder_input_tokens, decoder_input_tokens = load_next_batch()
print(f'Source: {batch["src_text"][0]}')
print(f'Target: {batch["tgt_text"][0]}')
sentence_len = encoder_input_tokens.index("[PAD]")

Source: "On the contrary, I have been to Devonshire."
Target: – Au contraire ! Je suis allé dans le Devonshire.


In [34]:
model_out = greedy_decode(model=model.to(device), src_input=batch["encoder_input"].to(device), src_mask=batch["encoder_mask"].to(device),
                        src_tokenizer=src_tokenizer, tgt_tokenizer=tgt_tokenizer, max_len=250,
                        device=device)

In [35]:
model_output_list = [tgt_tokenizer.id_to_token(idx) for idx in model_out.cpu().numpy()]
" ".join(model_output_list[1:-1])

'– Au contraire ! Je suis allé dans le Devonshire .'

In [36]:
layers = [0, 1, 2]
heads = [0, 1, 2, 3, 4, 5, 6, 7]

# Encoder Self-Attention
get_all_attention_maps("encoder", layers, heads, encoder_input_tokens, encoder_input_tokens, min(20, sentence_len))

alt.VConcatChart(...)

In [37]:
get_all_attention_maps("decoder", layers, heads, decoder_input_tokens, decoder_input_tokens, min(20, sentence_len))

alt.VConcatChart(...)

In [38]:
get_all_attention_maps("encoder-decoder", layers, heads, encoder_input_tokens, decoder_input_tokens, min(20, sentence_len))

alt.VConcatChart(...)